# 09_adipose_geometry

Selected source cells for review and reconstruction.

Source: `Adipose_combined_samples_CellSegmentation/Adipose_Combination_6(file separation_XYinformation).ipynb`

Uses curated combined adipose object and sample GeoJSON files.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
# Step 0: 環境/場所の確認
getwd()
list.files(pattern = "adipose_mergeobj_sketchHarmony_SCT_labelTransferred_k50_r020_Naming_260220\\.rds$")

suppressPackageStartupMessages({
  library(Seurat)
})


Source cell 2 (one-based); original execution count 2.

In [ ]:
# Step 1: 読み込み → sample列で4分割できるか確認
obj <- readRDS("adipose_mergeobj_sketchHarmony_SCT_labelTransferred_k50_r020_Naming_260220.rds")

colnames(obj[[]])        # sample列名を確認
table(obj$sample)        # Young_1/Young_2/Aged_1/Aged_2 が出るか

obj_list <- SplitObject(obj, split.by = "sample")
names(obj_list)

Source cell 3 (one-based); original execution count 3.

In [ ]:
## Step 2: 必要パッケージ（無ければ入れる）
pkgs <- c("Seurat", "dplyr", "ggplot2")

suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(ggplot2)
})

## Step 3: 4分割を保存（作った obj_list をそのまま使う）
saveRDS(obj_list[["Adipose_Young_1"]], file = "Adipose_Young_1_visiumHDsementation_ASSIGNEDonly.rds")
saveRDS(obj_list[["Adipose_Young_2"]], file = "Adipose_Young_2_visiumHDsementation_ASSIGNEDonly.rds")
saveRDS(obj_list[["Adipose_Aged_1"]],  file = "Adipose_Aged_1_visiumHDsementation_ASSIGNEDonly.rds")
saveRDS(obj_list[["Adipose_Aged_2"]],  file = "Adipose_Aged_2_visiumHDsementation_ASSIGNEDonly.rds")

Source cell 5 (one-based); original execution count 7.

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(dplyr)
  library(stringr)
  library(tibble)
})
sf::sf_use_s2(FALSE)

seu <- obj_list[["Adipose_Young_1"]]

gj <- "GeoJason/Adipose_Young_1/cell_segmentations.geojson"
stopifnot(file.exists(gj))

cells_sf <- st_read(gj, quiet = TRUE)

# GeoJSON 側 ID 列
id_col <- intersect(c("cell_id","barcode","id"), names(cells_sf))[1]
stopifnot(!is.na(id_col))

# 面積・重心（前回と同じ：seg_area_px + centroid_x/y を作る）
poly_geoms <- st_geometry(cells_sf)

invalid_idx <- which(!st_is_valid(poly_geoms))
if (length(invalid_idx)) {
  cells_sf[invalid_idx, ] <- st_make_valid(cells_sf[invalid_idx, ])
  poly_geoms <- st_geometry(cells_sf)
}

shoe_area <- function(m){
  if (is.null(m) || nrow(m) < 3) return(0)
  x <- m[,"X"]; y <- m[,"Y"]
  if (!(x[1] == x[length(x)] && y[1] == y[length(y)])) {
    x <- c(x, x[1]); y <- c(y, y[1])
  }
  0.5 * sum(x[-length(x)]*y[-1] - x[-1]*y[-length(y)])
}

calc_area_px <- function(g){
  if (is.null(g)) return(NA_real_)
  cc <- st_coordinates(g)
  if (is.null(cc) || nrow(cc) < 3) return(NA_real_)
  ring_col <- intersect(c("L3","L2","L1"), colnames(cc))[1]
  if (is.na(ring_col)) {
    return(abs(shoe_area(cc)))
  } else {
    cc_df <- as.data.frame(cc)
    rings <- split(cc_df, cc_df[[ring_col]], drop = TRUE)
    signed_sum <- 0
    for (rg in rings) signed_sum <- signed_sum + shoe_area(rg)
    return(abs(signed_sum))
  }
}

cells_sf$seg_area_px <- vapply(poly_geoms, calc_area_px, numeric(1))
cents <- st_centroid(poly_geoms)
coords <- st_coordinates(cents)
cells_sf$centroid_x <- coords[,1]
cells_sf$centroid_y <- coords[,2]

# Seurat 側 barcode → 数値ID抽出（あなたの今の形式は末尾 "000000002-1"）
barcodes <- colnames(seu)
id_num <- as.integer(str_extract(barcodes, "(?<=_)\\d+(?=-)"))
stopifnot(sum(is.na(id_num)) == 0)

map_df <- tibble(barcode = barcodes, !!id_col := id_num)

cell_meta <- cells_sf |>
  st_drop_geometry() |>
  dplyr::select(!!id_col, seg_area_px, centroid_x, centroid_y)

cell_meta_bar <- inner_join(map_df, cell_meta, by = id_col)

cell_meta_named <- column_to_rownames(cell_meta_bar, "barcode")
cell_meta_named <- as.data.frame(cell_meta_named)
cell_meta_named <- cell_meta_named[barcodes, , drop = FALSE]
stopifnot(identical(rownames(cell_meta_named), barcodes))

# ここが前回の踏襲ポイント：AddMetaData
seu <- AddMetaData(seu, metadata = cell_meta_named)

# 最小確認
tail(colnames(seu@meta.data))
c(sum(is.na(seu$centroid_x)), sum(is.na(seu$centroid_y)))

obj_list[["Adipose_Young_1"]] <- seu

Source cell 8 (one-based); original execution count 10.

In [ ]:
saveRDS(
  obj_list[["Adipose_Young_1"]],
  file = "Adipose_Young_1_visiumHDsementation_ASSIGNEDonly_withXY.rds"
)

Source cell 9 (one-based); original execution count 11.

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(dplyr)
  library(stringr)
  library(tibble)
})
sf::sf_use_s2(FALSE)

add_xy_from_geojson_like_prev <- function(seu, gj_path){
  stopifnot(file.exists(gj_path))
  cells_sf <- st_read(gj_path, quiet = TRUE)

  id_col <- intersect(c("cell_id","barcode","id"), names(cells_sf))[1]
  stopifnot(!is.na(id_col))

  poly_geoms <- st_geometry(cells_sf)
  invalid_idx <- which(!st_is_valid(poly_geoms))
  if (length(invalid_idx)) {
    cells_sf[invalid_idx, ] <- st_make_valid(cells_sf[invalid_idx, ])
    poly_geoms <- st_geometry(cells_sf)
  }

  shoe_area <- function(m){
    if (is.null(m) || nrow(m) < 3) return(0)
    x <- m[,"X"]; y <- m[,"Y"]
    if (!(x[1] == x[length(x)] && y[1] == y[length(y)])) {
      x <- c(x, x[1]); y <- c(y, y[1])
    }
    0.5 * sum(x[-length(x)]*y[-1] - x[-1]*y[-length(y)])
  }
  calc_area_px <- function(g){
    if (is.null(g)) return(NA_real_)
    cc <- st_coordinates(g)
    if (is.null(cc) || nrow(cc) < 3) return(NA_real_)
    ring_col <- intersect(c("L3","L2","L1"), colnames(cc))[1]
    if (is.na(ring_col)) {
      return(abs(shoe_area(cc)))
    } else {
      cc_df <- as.data.frame(cc)
      rings <- split(cc_df, cc_df[[ring_col]], drop = TRUE)
      signed_sum <- 0
      for (rg in rings) signed_sum <- signed_sum + shoe_area(rg)
      return(abs(signed_sum))
    }
  }

  cells_sf$seg_area_px <- vapply(poly_geoms, calc_area_px, numeric(1))
  cents <- st_centroid(poly_geoms)
  coords <- st_coordinates(cents)
  cells_sf$centroid_x <- coords[,1]
  cells_sf$centroid_y <- coords[,2]

  barcodes <- colnames(seu)
  id_num <- as.integer(str_extract(barcodes, "(?<=_)\\d+(?=-)"))
  stopifnot(sum(is.na(id_num)) == 0)

  map_df <- tibble(barcode = barcodes, !!id_col := id_num)
  cell_meta <- cells_sf |>
    st_drop_geometry() |>
    dplyr::select(!!id_col, seg_area_px, centroid_x, centroid_y)

  cell_meta_bar <- inner_join(map_df, cell_meta, by = id_col)
  cell_meta_named <- column_to_rownames(cell_meta_bar, "barcode")
  cell_meta_named <- as.data.frame(cell_meta_named)
  cell_meta_named <- cell_meta_named[barcodes, , drop = FALSE]
  stopifnot(identical(rownames(cell_meta_named), barcodes))

  AddMetaData(seu, metadata = cell_meta_named)
}

# --- 残り3サンプルに適用 ---
targets <- c("Adipose_Young_2", "Adipose_Aged_1", "Adipose_Aged_2")
for (s in targets) {
  gj <- file.path("GeoJason", s, "cell_segmentations.geojson")
  obj_list[[s]] <- add_xy_from_geojson_like_prev(obj_list[[s]], gj)

  # 前回と同じ最小確認（seg_area_px NAが無いこと）
  cat("\n", s, " NA(seg_area_px)=", sum(is.na(obj_list[[s]]$seg_area_px)), "\n", sep = "")
}

# --- 保存（withXY）---
saveRDS(obj_list[["Adipose_Young_2"]], file = "Adipose_Young_2_visiumHDsementation_ASSIGNEDonly_withXY.rds")
saveRDS(obj_list[["Adipose_Aged_1"]],  file = "Adipose_Aged_1_visiumHDsementation_ASSIGNEDonly_withXY.rds")
saveRDS(obj_list[["Adipose_Aged_2"]],  file = "Adipose_Aged_2_visiumHDsementation_ASSIGNEDonly_withXY.rds")

Source cell 10 (one-based); original execution count 12.

In [ ]:
sapply(obj_list[c("Adipose_Young_1","Adipose_Young_2","Adipose_Aged_1","Adipose_Aged_2")],
       function(o) c(NAx=sum(is.na(o$centroid_x)), NAy=sum(is.na(o$centroid_y))))

Source cell 13 (one-based); original execution count 14.

In [ ]:
library(dplyr)

samples <- c("Adipose_Young_1","Adipose_Young_2","Adipose_Aged_1","Adipose_Aged_2")

for (s in samples) {
  seu <- obj_list[[s]]
  md <- seu@meta.data

  meta_out <- data.frame(
    barcode  = colnames(seu),
    celltype = md$Merged_cellname,
    sample   = md$sample,
    x        = md$centroid_x,
    y        = md$centroid_y,
    row.names = NULL
  )

  fn <- paste0(s, "_meta.csv")
  write.csv(meta_out, file = fn, quote = FALSE, row.names = FALSE)
  cat("wrote:", fn, " n=", nrow(meta_out), "\n")
}